# Contextual Retrieval로 RAG 향상하기

> 참고: 여러 dataset의 추가 성능 평가를 포함한 Contextual Retrieval의 배경 정보는 함께 제공하는 [blog 게시물](https://www.anthropic.com/news/contextual-retrieval)을 읽는 것을 권장합니다.

Retrieval Augmented Generation (RAG)은 Claude가 응답할 때 내부 knowledge base, codebase 또는 다른 문서 corpus를 활용할 수 있게 합니다. 기업들은 고객 지원, 내부 회사 문서에 대한 Q&A, 재무 및 법률 분석, 코드 생성 등 다양한 workflow를 개선하기 위해 RAG 애플리케이션을 점점 더 많이 구축하고 있습니다.

[별도 가이드](https://github.com/anthropics/anthropic-cookbook/blob/main/capabilities/retrieval_augmented_generation/guide.ipynb)에서는 기본 retrieval 시스템 설정, 성능 평가 방법 및 성능을 개선하는 몇 가지 기법을 설명했습니다. 이 가이드에서는 retrieval 성능을 개선하는 기법인 Contextual Embedding을 소개합니다.

기존 RAG에서는 효율적인 retrieval을 위해 문서를 작은 chunk로 나누는 것이 일반적입니다. 여러 애플리케이션에서 잘 작동하지만 개별 chunk에 충분한 context가 없으면 문제가 생길 수 있습니다. Contextual Embedding은 embedding 전에 각 chunk에 관련 context를 추가하여 이를 해결합니다. 각 embedding된 chunk의 품질을 높여 더 정확한 retrieval 및 더 좋은 전반적 성능을 제공합니다. 테스트한 모든 데이터 출처의 평균에서 Contextual Embedding은 상위 20개 chunk의 retrieval 실패율을 35% 줄였습니다.

같은 chunk별 context를 BM25 검색에도 사용하여 retrieval 성능을 더 개선할 수 있습니다. "Contextual BM25" 절에서 소개합니다.

이 가이드에서는 codebase 9개의 dataset을 knowledge base로 사용하여 Contextual Retrieval 시스템을 구축하고 최적화하는 방법을 보여줍니다. 다음을 살펴봅니다:

1) 성능 baseline을 수립할 기본 retrieval pipeline 설정.

2) Contextual Embedding의 개념, 작동 이유 및 prompt caching이 production 사용 사례에 실용적으로 만드는 방식.

3) Contextual Embedding 구현 및 성능 개선 제시.

4) Contextual BM25: *Contextual* BM25 hybrid search로 성능 개선.

5) Reranking으로 성능 개선,

### 평가 Metric 및 Dataset:

기본적인 문자 분할 방식으로 미리 chunk로 나눈 codebase 9개의 dataset을 사용합니다. 평가 dataset에는 각각 'golden chunk'가 있는 query 248개가 포함됩니다. Pass@k라는 metric으로 성능을 평가합니다. Pass@k는 각 query에서 retrieval한 첫 k개 문서에 'golden document'가 있는지 확인합니다. 이 사례에서 Contextual Embedding은 Pass@10 성능을 약 87% --> 약 95%로 개선하는 데 도움이 되었습니다.

코드 파일과 chunk는 `data/codebase_chunks.json`, 평가 dataset은 `data/evaluation_set.jsonl`에서 찾을 수 있습니다

#### 추가 참고 사항:

Prompt caching은 이 retrieval 방법의 비용 관리에 도움이 됩니다. 이 기능은 현재 Anthropic first-party API에서 이용 가능하며 AWS Bedrock 및 GCP Vertex의 third-party partner 환경에도 곧 제공될 예정입니다. 많은 고객이 RAG 솔루션을 구축할 때 AWS Knowledge Bases 및 GCP Vertex AI API를 활용하며, 이 방법은 조금만 맞춤화하면 두 platform 모두에서 사용할 수 있습니다. 안내가 필요하면 Anthropic이나 AWS/GCP account team에 문의하는 것을 고려하세요!

Bedrock에서 더 쉽게 사용하도록 AWS team이 각 문서에 context를 추가하는 Lambda 함수 구현 코드를 제공했습니다. 이 Lambda 함수를 배포하면 [Bedrock Knowledge Base](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base-create.html) 설정 시 custom chunking 옵션으로 선택할 수 있습니다. 코드는 `contextual-rag-lambda-function`에 있습니다. 주 lambda 함수 코드는 `lambda_function.py`입니다.

## 목차

1) 설정

2) 기본 RAG

3) Contextual Embedding

4) Contextual BM25

5) Reranking

## 설정

이 가이드를 시작하기 전에 다음을 준비하세요:

**기술 역량:**
- 중급 Python programming
- RAG (Retrieval Augmented Generation)의 기본 이해
- Vector database와 embedding에 대한 이해
- 기본 command-line 사용 능력

**시스템 요구 사항:**
- Python 3.8+
- 설치되어 실행 중인 Docker(선택 사항, BM25 검색용)
- 사용 가능한 RAM 4GB+
- Vector database용 disk 공간 약 5-10 GB

**API 접근:**
- [Anthropic API key](https://console.anthropic.com/)(free tier로 충분)
- [Voyage AI API key](https://www.voyageai.com/)
- [Cohere API key](https://cohere.com/)

**시간 및 비용:**
- 예상 완료 시간: 30-45분
- API 비용: 전체 dataset 실행에 약 $5-10

### 라이브러리 

다음 라이브러리가 필요합니다:

1) `anthropic` - Claude와 상호 작용

2) `voyageai` - 고품질 embedding 생성

3) `cohere` - reranking

4) 성능 좋은 BM25 검색을 위한 `elasticsearch`

3) 데이터 조작 및 시각화를 위한 `pandas`, `numpy`, `matplotlib`, `scikit-learn`

### 환경 변수 

다음 환경 변수가 설정되어 있는지 확인하세요:

```
- VOYAGE_API_KEY
- ANTHROPIC_API_KEY
- COHERE_API_KEY
```

In [6]:
%%capture
%pip install --upgrade anthropic voyageai cohere elasticsearch pandas numpy

새 모델이 출시될 때 쉽게 바꿀 수 있도록 모델 이름을 미리 정의합니다

In [ ]:
MODEL_NAME = "claude-haiku-4-5"

Context 설명을 생성하는 데 사용할 Anthropic client부터 초기화하겠습니다.

In [5]:
import os

import anthropic

client = anthropic.Anthropic(
    # 기본값이며 생략 가능
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

## Vector DB Class 초기화

Embedding 저장 및 similarity search를 처리하는 VectorDB class를 만들겠습니다. 이 class는 RAG pipeline에서 세 가지 핵심 기능을 수행합니다:

1. **Embedding 생성**: Voyage AI embedding 모델로 텍스트 chunk를 vector 표현으로 변환
2. **저장 및 Caching**: 다시 계산하지 않도록 embedding을 disk에 저장(시간 및 API 비용 절약)
3. **Similarity Search**: Cosine similarity로 주어진 query의 가장 관련성 높은 chunk를 retrieval


이 가이드에서는 pickle serialization을 사용하는 단순한 메모리 내 vector database를 사용합니다. 코드를 쉽게 이해할 수 있으며 외부 dependency가 필요 없습니다. Class는 생성 후 embedding을 자동으로 disk에 저장하므로 embedding 비용을 한 번만 지불합니다. 

Production 용도로는 호스팅된 vector database 솔루션을 고려하세요.

아래 VectorDB class는 production 솔루션에서 사용할 것과 같은 interface 패턴을 따라 나중에 쉽게 교체할 수 있습니다. 주요 기능에는 batch 처리(한 번에 128 chunk), tqdm으로 진행 추적, 평가 중 반복 검색 속도를 높이는 query caching이 있습니다.

In [ ]:
import json
import pickle
from typing import Any

import numpy as np
import voyageai
from tqdm import tqdm


class VectorDB:
    def __init__(self, name: str, api_key=None):
        if api_key is None:
            api_key = os.getenv("VOYAGE_API_KEY")
        self.client = voyageai.Client(api_key=api_key)
        self.name = name
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = f"./data/{name}/vector_db.pkl"

    def load_data(self, dataset: list[dict[str, Any]]):
        if self.embeddings and self.metadata:
            print("Vector database is already loaded. Skipping data loading.")
            return
        if os.path.exists(self.db_path):
            print("Loading vector database from disk.")
            self.load_db()
            return

        texts_to_embed = []
        metadata = []
        total_chunks = sum(len(doc["chunks"]) for doc in dataset)

        with tqdm(total=total_chunks, desc="Processing chunks") as pbar:
            for doc in dataset:
                for chunk in doc["chunks"]:
                    texts_to_embed.append(chunk["content"])
                    metadata.append(
                        {
                            "doc_id": doc["doc_id"],
                            "original_uuid": doc["original_uuid"],
                            "chunk_id": chunk["chunk_id"],
                            "original_index": chunk["original_index"],
                            "content": chunk["content"],
                        }
                    )
                    pbar.update(1)

        self._embed_and_store(texts_to_embed, metadata)
        self.save_db()

        print(f"Vector database loaded and saved. Total chunks processed: {len(texts_to_embed)}")

    def _embed_and_store(self, texts: list[str], data: list[dict[str, Any]]):
        batch_size = 128
        with tqdm(total=len(texts), desc="Embedding chunks") as pbar:
            result = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                batch_result = self.client.embed(batch, model="voyage-2").embeddings
                result.extend(batch_result)
                pbar.update(len(batch))

        self.embeddings = result
        self.metadata = data

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("No data loaded in the vector database.")

        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]

        top_results = []
        for idx in top_indices:
            result = {
                "metadata": self.metadata[idx],
                "similarity": float(similarities[idx]),
            }
            top_results.append(result)

        return top_results

    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": json.dumps(self.query_cache),
        }
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)

    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError(
                "Vector database file not found. Use load_data to create a new database."
            )
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = json.loads(data["query_cache"])

이제 이 class로 dataset을 로드할 수 있습니다

In [12]:
# 변환된 dataset 로드
with open("data/codebase_chunks.json") as f:
    transformed_dataset = json.load(f)

# VectorDB 초기화
base_db = VectorDB("base_db")

# 데이터 로드 및 처리
base_db.load_data(transformed_dataset)

Embedding chunks: 100%|██████████| 737/737 [00:42<00:00, 17.28it/s]

Vector database loaded and saved. Total chunks processed: 737


## 기본 RAG

먼저 최소한의 방식으로 기본 RAG pipeline을 설정하겠습니다. 업계에서는 때때로 'Naive RAG'라고 부릅니다. 기본 RAG pipeline은 다음 3단계를 포함합니다:

1) 각 소제목의 내용만 포함하도록 제목별로 문서를 chunk로 나누기

2) 각 문서 embedding

3) Cosine similarity로 query에 답할 문서 retrieval

In [26]:
import json
from collections.abc import Callable
from typing import Any

from tqdm import tqdm


def load_jsonl(file_path: str) -> list[dict[str, Any]]:
    """JSONL 파일을 로드하고 dictionary 목록을 반환합니다."""
    with open(file_path) as file:
        return [json.loads(line) for line in file]


def evaluate_retrieval(
    queries: list[dict[str, Any]], retrieval_function: Callable, db, k: int = 20
) -> dict[str, float]:
    total_score = 0
    total_queries = len(queries)

    for query_item in tqdm(queries, desc="Evaluating retrieval"):
        query = query_item["query"]
        golden_chunk_uuids = query_item["golden_chunk_uuids"]

        # 모든 golden chunk 내용 찾기
        golden_contents = []
        for doc_uuid, chunk_index in golden_chunk_uuids:
            golden_doc = next(
                (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid), None
            )
            if not golden_doc:
                print(f"Warning: Golden document not found for UUID {doc_uuid}")
                continue

            golden_chunk = next(
                (chunk for chunk in golden_doc["chunks"] if chunk["index"] == chunk_index), None
            )
            if not golden_chunk:
                print(
                    f"Warning: Golden chunk not found for index {chunk_index} in document {doc_uuid}"
                )
                continue

            golden_contents.append(golden_chunk["content"].strip())

        if not golden_contents:
            print(f"Warning: No golden contents found for query: {query}")
            continue

        retrieved_docs = retrieval_function(query, db, k=k)

        # 상위 k개 retrieval 문서의 golden chunk 수 계산
        chunks_found = 0
        for golden_content in golden_contents:
            for doc in retrieved_docs[:k]:
                retrieved_content = (
                    doc["metadata"]
                    .get("original_content", doc["metadata"].get("content", ""))
                    .strip()
                )
                if retrieved_content == golden_content:
                    chunks_found += 1
                    break

        query_score = chunks_found / len(golden_contents)
        total_score += query_score

    average_score = total_score / total_queries
    pass_at_n = average_score * 100
    return {"pass_at_n": pass_at_n, "average_score": average_score, "total_queries": total_queries}


def retrieve_base(query: str, db, k: int = 20) -> list[dict[str, Any]]:
    """
    VectorDB 또는 ContextualVectorDB로 관련 문서를 retrieval합니다.

    :param query: Query 문자열
    :param db: VectorDB 또는 ContextualVectorDB instance
    :param k: Retrieval할 상위 결과 수
    :return: Retrieval된 문서 목록
    """
    return db.search(query, k=k)


def evaluate_db(db, original_jsonl_path: str, k):
    # Query 및 ground truth용 원본 JSONL 데이터 로드
    original_data = load_jsonl(original_jsonl_path)

    # Retrieval 평가
    results = evaluate_retrieval(original_data, retrieve_base, db, k)
    return results


def evaluate_and_display(db, jsonl_path: str, k_values: list[int] = None, db_name: str = ""):
    """
    여러 k 값에서 retrieval 성능을 평가하고 형식을 정리한 결과를 표시합니다.

    Args:
        db: Vector database instance(VectorDB 또는 ContextualVectorDB)
        jsonl_path: 평가 dataset 경로
        k_values: 평가할 k 값 목록(기본값: [5, 10, 20])
        db_name: 평가하는 database의 선택적 이름

    Returns:
        k 값을 결과에 mapping하는 dict
    """
    if k_values is None:
        k_values = [5, 10, 20]
    results = {}

    print(f"{'=' * 60}")
    if db_name:
        print(f"Evaluation Results: {db_name}")
    else:
        print("Evaluation Results")
    print(f"{'=' * 60}\n")

    for k in k_values:
        print(f"Evaluating Pass@{k}...")
        results[k] = evaluate_db(db, jsonl_path, k)
        print()  # 평가 사이 공백 추가

    # 요약 표 출력
    print(f"{'=' * 60}")
    print(f"{'Metric':<15} {'Pass Rate':<15} {'Score':<15}")
    print(f"{'-' * 60}")
    for k in k_values:
        pass_rate = f"{results[k]['pass_at_n']:.2f}%"
        score = f"{results[k]['average_score']:.4f}"
        print(f"{'Pass@' + str(k):<15} {pass_rate:<15} {score:<15}")
    print(f"{'=' * 60}\n")

    return results

이제 기본 RAG 시스템을 평가하여 baseline 성능을 수립하겠습니다. k=5, 10, 20에서 golden chunk가 상위 retrieval 결과에 얼마나 나타나는지 테스트합니다. 개선을 측정할 benchmark를 제공합니다.


In [ ]:
results = evaluate_and_display(
    base_db, "data/evaluation_set.jsonl", k_values=[5, 10, 20], db_name="Baseline RAG"
)

Evaluation Results: Contextual Embeddings

Evaluating Pass@5...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 65.26it/s]



Evaluating Pass@10...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.87it/s]



Evaluating Pass@20...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.72it/s]


Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          80.92%          0.8092         
Pass@10         87.15%          0.8715         
Pass@20         90.06%          0.9006         



이 결과는 baseline RAG 성능을 보여줍니다. 시스템은 상위 5개 결과에서 81%의 경우 올바른 chunk를 retrieval하며 상위 10개에서는 87%, 상위 20개에서는 90%로 개선됩니다.

## Contextual Embedding

기본 RAG에서는 개별 chunk를 따로 embedding할 때 context가 부족한 경우가 많습니다. Contextual Embedding은 Claude로 각 chunk를 source document 안에 "위치시키는" 짧은 설명을 생성하여 해결합니다. 그런 다음 이 context와 함께 chunk를 embedding하여 더 풍부한 vector 표현을 만듭니다. 

Codebase dataset의 각 chunk에 대해 chunk와 전체 source file을 모두 Claude에 전달합니다. Claude는 chunk의 내용과 전체 파일에서의 위치를 간결하게 설명합니다. Embedding 전에 이 context를 chunk 앞에 붙입니다.


### 비용 및 Latency 고려 사항

**언제 비용이 발생하나요?** Contextualization은 query마다가 아니라 ingestion 시 한 번 수행됩니다. 검색마다 latency를 추가하는 HyDE (hypothetical document embeddings) 같은 기법과 달리 contextual embedding은 vector database 구축 시 일회성 비용입니다. Prompt caching으로 실용성이 높아집니다. 같은 문서의 모든 chunk를 순서대로 처리하므로 prompt caching으로 크게 절약할 수 있습니다.

1. 첫 chunk: 전체 문서를 cache에 기록(소액의 할증 지불)
2. 이후 chunk: Cache에서 문서 읽기(해당 token에 90% 할인)
3. Cache는 5분 유지되어 문서의 모든 chunk를 처리하기에 충분함

**비용 예제**: 8k-token 문서의 800-token chunk에 생성한 context 100 token을 붙이면 총 비용은 문서 token 백만 개당 $1.02입니다. 아래 코드를 실행하면 로그에서 cache 절감 효과를 볼 수 있습니다.

**참고:** 일부 embedding 모델은 input token 제한이 고정되어 있습니다. Contextual embedding의 성능이 더 나쁘면 contextualized chunk가 잘리고 있을 수 있습니다. 더 큰 context window를 가진 embedding 모델 사용을 고려하세요.

--- 

Chunk 하나의 context를 생성하여 contextual embedding의 작동 예제를 살펴보겠습니다. Claude로 위치를 설명하는 context를 만들며 prompt caching metric도 실제로 확인할 수 있습니다.


In [ ]:
DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""

CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""


def situate_context(doc: str, chunk: str) -> str:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc),
                        "cache_control": {"type": "ephemeral"},  # 전체 문서에 prompt caching 사용
                    },
                    {
                        "type": "text",
                        "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk),
                    },
                ],
            }
        ],
    )
    return response


jsonl_data = load_jsonl("data/evaluation_set.jsonl")
# 사용 예제
doc_content = jsonl_data[0]["golden_documents"][0]["content"]
chunk_content = jsonl_data[0]["golden_chunks"][0]["content"]

response = situate_context(doc_content, chunk_content)
print(f"Situated context: {response.content[0].text}")
print("-" * 10)
# Cache 성능 metric 출력
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Cache creation input tokens: {response.usage.cache_creation_input_tokens}")
print(f"Cache read input tokens: {response.usage.cache_read_input_tokens}")

Situated context: This chunk contains the module documentation and initial struct definition for a differential fuzzing executor. It introduces the `DiffExecutor` struct that wraps two executors (primary and secondary) to run them sequentially with the same input, comparing their behavior for differential testing. The chunk establishes the core data structure and imports needed for the differential fuzzing implementation.
----------
Input tokens: 3412
Output tokens: 76
Cache creation input tokens: 0
Cache read input tokens: 0


### Contextual Vector Database 구축

개별 chunk의 context 설명을 생성하는 방법을 보았으므로 전체 dataset을 처리하도록 확장하겠습니다. 아래 `ContextualVectorDB` class는 기본 `VectorDB`를 확장하여 ingestion 중 자동 contextualization을 수행합니다.

**주요 기능:**

- **병렬 처리**: ThreadPoolExecutor로 여러 chunk를 동시에 contextualization(thread 수 설정 가능)
- **자동 prompt caching**: Cache hit을 최대화하도록 문서별로 chunk 처리
- **Token 추적**: Cache 성능 모니터링 및 실제 비용 절감 계산
- **Persistent storage**: Embedding 및 contextualized metadata를 disk에 저장

실행할 때 token 사용 통계에 주목하세요. Input token의 70-80%를 cache에서 읽으며 prompt caching의 극적인 비용 절감을 보여줍니다. 737개 chunk dataset에서는 약 $15가 될 ingestion job을 약 $3로 줄입니다.

In [ ]:
import json
import os
import pickle
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any

import anthropic
import numpy as np
import voyageai
from tqdm import tqdm


class ContextualVectorDB:
    def __init__(self, name: str, voyage_api_key=None, anthropic_api_key=None):
        if voyage_api_key is None:
            voyage_api_key = os.getenv("VOYAGE_API_KEY")
        if anthropic_api_key is None:
            anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

        self.voyage_client = voyageai.Client(api_key=voyage_api_key)
        self.anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
        self.name = name
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = f"./data/{name}/contextual_vector_db.pkl"

        self.token_counts = {"input": 0, "output": 0, "cache_read": 0, "cache_creation": 0}
        self.token_lock = threading.Lock()

    def situate_context(self, doc: str, chunk: str) -> tuple[str, Any]:
        DOCUMENT_CONTEXT_PROMPT = """
        <document>
        {doc_content}
        </document>
        """

        CHUNK_CONTEXT_PROMPT = """
        Here is the chunk we want to situate within the whole document
        <chunk>
        {chunk_content}
        </chunk>

        Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
        Answer only with the succinct context and nothing else.
        """

        response = self.anthropic_client.messages.create(
            model=MODEL_NAME,
            max_tokens=1000,
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc),
                            "cache_control": {
                                "type": "ephemeral"
                            },  # 전체 문서에 prompt caching 사용
                        },
                        {
                            "type": "text",
                            "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk),
                        },
                    ],
                },
            ],
            extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"},
        )
        return response.content[0].text, response.usage

    def load_data(self, dataset: list[dict[str, Any]], parallel_threads: int = 1):
        if self.embeddings and self.metadata:
            print("Vector database is already loaded. Skipping data loading.")
            return
        if os.path.exists(self.db_path):
            print("Loading vector database from disk.")
            self.load_db()
            return

        texts_to_embed = []
        metadata = []
        total_chunks = sum(len(doc["chunks"]) for doc in dataset)

        def process_chunk(doc, chunk):
            # 각 chunk의 context 생성
            contextualized_text, usage = self.situate_context(doc["content"], chunk["content"])
            with self.token_lock:
                self.token_counts["input"] += usage.input_tokens
                self.token_counts["output"] += usage.output_tokens
                self.token_counts["cache_read"] += usage.cache_read_input_tokens
                self.token_counts["cache_creation"] += usage.cache_creation_input_tokens

            return {
                # 원래 텍스트 chunk에 context 추가
                "text_to_embed": f"{contextualized_text}\n\n{chunk['content']}",
                "metadata": {
                    "doc_id": doc["doc_id"],
                    "original_uuid": doc["original_uuid"],
                    "chunk_id": chunk["chunk_id"],
                    "original_index": chunk["original_index"],
                    "original_content": chunk["content"],
                    "contextualized_content": contextualized_text,
                },
            }

        print(f"Processing {total_chunks} chunks with {parallel_threads} threads")
        with ThreadPoolExecutor(max_workers=parallel_threads) as executor:
            futures = []
            for doc in dataset:
                for chunk in doc["chunks"]:
                    futures.append(executor.submit(process_chunk, doc, chunk))

            for future in tqdm(as_completed(futures), total=total_chunks, desc="Processing chunks"):
                result = future.result()
                texts_to_embed.append(result["text_to_embed"])
                metadata.append(result["metadata"])

        self._embed_and_store(texts_to_embed, metadata)
        self.save_db()

        # Token 사용량 logging
        print(
            f"Contextual Vector database loaded and saved. Total chunks processed: {len(texts_to_embed)}"
        )
        print(f"Total input tokens without caching: {self.token_counts['input']}")
        print(f"Total output tokens: {self.token_counts['output']}")
        print(f"Total input tokens written to cache: {self.token_counts['cache_creation']}")
        print(f"Total input tokens read from cache: {self.token_counts['cache_read']}")

        total_tokens = (
            self.token_counts["input"]
            + self.token_counts["cache_read"]
            + self.token_counts["cache_creation"]
        )
        savings_percentage = (
            (self.token_counts["cache_read"] / total_tokens) * 100 if total_tokens > 0 else 0
        )
        print(
            f"Total input token savings from prompt caching: {savings_percentage:.2f}% of all input tokens used were read from cache."
        )
        print("Tokens read from cache come at a 90 percent discount!")

    # 여기서는 embedding에 Voyage AI를 사용합니다. 자세한 내용: https://docs.voyageai.com/docs/embeddings
    def _embed_and_store(self, texts: list[str], data: list[dict[str, Any]]):
        batch_size = 128
        result = [
            self.voyage_client.embed(texts[i : i + batch_size], model="voyage-2").embeddings
            for i in range(0, len(texts), batch_size)
        ]
        self.embeddings = [embedding for batch in result for embedding in batch]
        self.metadata = data

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.voyage_client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("No data loaded in the vector database.")

        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]

        top_results = []
        for idx in top_indices:
            result = {
                "metadata": self.metadata[idx],
                "similarity": float(similarities[idx]),
            }
            top_results.append(result)
        return top_results

    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": json.dumps(self.query_cache),
        }
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)

    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError(
                "Vector database file not found. Use load_data to create a new database."
            )
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = json.loads(data["query_cache"])

In [22]:
# 변환된 dataset 로드
with open("data/codebase_chunks.json") as f:
    transformed_dataset = json.load(f)

# ContextualVectorDB 초기화
contextual_db = ContextualVectorDB("my_contextual_db")

# 데이터 로드 및 처리
# 참고: 더 빠른 실행을 위해 병렬 thread 수를 늘리거나 API rate limit 도달이 우려되면 줄이는 것을 고려하세요
contextual_db.load_data(transformed_dataset, parallel_threads=5)

Processing 737 chunks with 5 threads


Processing chunks: 100%|██████████| 737/737 [05:32<00:00,  2.22it/s]


Contextual Vector database loaded and saved. Total chunks processed: 737
Total input tokens without caching: 1223730
Total output tokens: 58161
Total input tokens written to cache: 176079
Total input tokens read from cache: 2267069
Total input token savings from prompt caching: 61.83% of all input tokens used were read from cache.
Tokens read from cache come at a 90 percent discount!


이 수치는 contextual embedding에 대한 prompt caching의 힘을 보여줍니다:

- Codebase 파일 9개에서 **737개 chunk** 처리
- **Input token의 61.83%**를 cache에서 읽음(90% 할인된 2.27M token)
- Caching이 없으면 input token 비용 **약 $9.20**
- Caching을 사용하면 실제 비용 **약 $2.85**로 감소(69% 절감)

Cache hit rate는 각 문서의 chunk 수에 따라 달라집니다. Chunk가 많은 파일은 전체 문서를 한 번 cache에 기록한 뒤 해당 파일의 각 chunk에서 반복하여 읽으므로 더 큰 이점을 얻습니다. 따라서 cache 효율을 최대화하려면 chunk를 무작위로 섞는 대신 문서를 순서대로 처리하는 것이 중요합니다.

이제 contextualization이 baseline 대비 retrieval 성능을 얼마나 개선하는지 평가하겠습니다.

In [28]:
results = evaluate_and_display(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual Embeddings",
)

Evaluation Results: Contextual Embeddings

Evaluating Pass@5...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.58it/s]



Evaluating Pass@10...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.37it/s]



Evaluating Pass@20...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.14it/s]


Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          88.12%          0.8812         
Pass@10         92.34%          0.9234         
Pass@20         94.29%          0.9429         



Embedding 전에 각 chunk에 context를 추가하여 모든 k 값에서 retrieval 실패를 **약 30-40%** 줄였습니다. 상위 retrieval chunk에 관련 없는 결과가 줄어들어, 최종 응답 생성을 위해 Claude에 전달할 때 더 좋은 답을 얻을 수 있다는 의미입니다.

개선은 precision이 가장 중요한 Pass@5에서 가장 뚜렷합니다. Contextualized chunk가 더 자주 retrieval될 뿐 아니라 관련이 있을 때 더 높은 순위를 얻는다는 점을 시사합니다.

## Contextual BM25: Hybrid Search

Contextual embedding만으로 Pass@10을 87%에서 92%로 개선했습니다. **Contextual BM25**를 사용하여 semantic search와 keyword 기반 검색을 결합하면 성능을 더 높일 수 있습니다. Retrieval 실패율을 더 줄이는 hybrid 방식입니다.

### Hybrid Search를 사용하는 이유

Semantic search는 의미와 context 이해에 뛰어나지만 정확한 keyword 일치를 놓칠 수 있습니다. BM25(확률적 keyword ranking algorithm)는 특정 용어를 찾는 데 뛰어나지만 의미 이해가 부족합니다. 둘을 결합하면 두 방식의 장점을 모두 얻습니다:

- **Semantic search**: 개념적 유사성과 paraphrase 포착
- **BM25**: 정확한 전문용어, 함수 이름 및 특정 구문 포착
- **Reciprocal Rank Fusion**: 두 출처의 결과를 지능적으로 병합

### BM25란 무엇인가요?

BM25는 문서 길이와 term saturation을 고려하여 TF-IDF를 개선하는 확률적 ranking function입니다. Keyword 관련성 ranking에 효과적이므로 Elasticsearch를 포함한 production search engine에서 널리 사용합니다. 기술적인 세부 사항은 [이 blog 게시물](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables)을 참조하세요.

원시 chunk 내용만 검색하는 대신 chunk*와* 앞서 생성한 context 설명을 모두 검색합니다. 따라서 BM25는 원문 또는 설명 context의 keyword에 일치할 수 있습니다.

### 설정: Elasticsearch 실행

아래 코드를 실행하기 전에 Elasticsearch를 로컬에서 실행해야 합니다. 가장 쉬운 방법은 Docker입니다:

```bash
docker run -d --name elasticsearch -p 9200:9200 -p 9300:9300 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  elasticsearch:9.2.0
```

## 문제 해결:
- 실행 여부 확인: docker ps | grep elasticsearch
- Port 9200 사용 중인 경우: docker stop elasticsearch && docker rm elasticsearch
- 문제 발생 시 로그 확인: docker logs elasticsearch

## Hybrid Search 작동 방식

아래 retrieve_advanced 함수는 3단계 과정을 구현합니다:

1. 후보 retrieval: Semantic search와 BM25에서 각각 상위 150개 결과 조회
2. Score fusion: 가중 Reciprocal Rank Fusion으로 ranking 결합
    - 기본값: Semantic search 80%, BM25 20% 가중치
    - 가중치를 조정할 수 있으므로 사용 사례에 맞게 실험하여 최적화
3. Top-k 반환: Fusion 후 가장 높은 score의 결과 선택

가중치 시스템으로 데이터 특성에 따라 의미 이해와 keyword precision 간 균형을 맞출 수 있습니다.

In [ ]:
import json
import os
from typing import Any

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from tqdm import tqdm


class ElasticsearchBM25:
    def __init__(self, index_name: str = "contextual_bm25_index"):
        self.es_client = Elasticsearch("http://localhost:9200")
        self.index_name = index_name
        self.create_index()

    def create_index(self):
        index_settings = {
            "settings": {
                "analysis": {"analyzer": {"default": {"type": "english"}}},
                "similarity": {"default": {"type": "BM25"}},
                "index.queries.cache.enabled": False,
            },
            "mappings": {
                "properties": {
                    "content": {"type": "text", "analyzer": "english"},
                    "contextualized_content": {"type": "text", "analyzer": "english"},
                    "doc_id": {"type": "keyword", "index": False},
                    "chunk_id": {"type": "keyword", "index": False},
                    "original_index": {"type": "integer", "index": False},
                }
            },
        }

        # 이 줄 변경 - 'body=' parameter 제거
        if not self.es_client.indices.exists(index=self.index_name):
            self.es_client.indices.create(
                index=self.index_name,
                settings=index_settings["settings"],
                mappings=index_settings["mappings"],
            )
            print(f"Created index: {self.index_name}")

    def index_documents(self, documents: list[dict[str, Any]]):
        actions = [
            {
                "_index": self.index_name,
                "_source": {
                    "content": doc["original_content"],
                    "contextualized_content": doc["contextualized_content"],
                    "doc_id": doc["doc_id"],
                    "chunk_id": doc["chunk_id"],
                    "original_index": doc["original_index"],
                },
            }
            for doc in documents
        ]
        success, _ = bulk(self.es_client, actions)
        self.es_client.indices.refresh(index=self.index_name)
        return success

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        self.es_client.indices.refresh(index=self.index_name)

        # 변경 - 'body=' 제거 및 query 직접 전달
        response = self.es_client.search(
            index=self.index_name,
            query={
                "multi_match": {
                    "query": query,
                    "fields": ["content", "contextualized_content"],
                }
            },
            size=k,
        )

        return [
            {
                "doc_id": hit["_source"]["doc_id"],
                "original_index": hit["_source"]["original_index"],
                "content": hit["_source"]["content"],
                "contextualized_content": hit["_source"]["contextualized_content"],
                "score": hit["_score"],
            }
            for hit in response["hits"]["hits"]
        ]


def create_elasticsearch_bm25_index(db: ContextualVectorDB):
    es_bm25 = ElasticsearchBM25()
    es_bm25.index_documents(db.metadata)
    return es_bm25


def retrieve_advanced(
    query: str,
    db: ContextualVectorDB,
    es_bm25: ElasticsearchBM25,
    k: int,
    semantic_weight: float = 0.8,
    bm25_weight: float = 0.2,
):
    num_chunks_to_recall = 150

    # Semantic search
    semantic_results = db.search(query, k=num_chunks_to_recall)
    ranked_chunk_ids = [
        (result["metadata"]["doc_id"], result["metadata"]["original_index"])
        for result in semantic_results
    ]

    # Elasticsearch를 사용한 BM25 검색
    bm25_results = es_bm25.search(query, k=num_chunks_to_recall)
    ranked_bm25_chunk_ids = [
        (result["doc_id"], result["original_index"]) for result in bm25_results
    ]

    # 결과 결합
    chunk_ids = list(set(ranked_chunk_ids + ranked_bm25_chunk_ids))
    chunk_id_to_score = {}

    # 가중치로 초기 scoring
    for chunk_id in chunk_ids:
        score = 0
        if chunk_id in ranked_chunk_ids:
            index = ranked_chunk_ids.index(chunk_id)
            score += semantic_weight * (1 / (index + 1))  # Semantic용 가중 1/n scoring
        if chunk_id in ranked_bm25_chunk_ids:
            index = ranked_bm25_chunk_ids.index(chunk_id)
            score += bm25_weight * (1 / (index + 1))  # BM25용 가중 1/n scoring
        chunk_id_to_score[chunk_id] = score

    # Score 내림차순으로 chunk ID 정렬
    sorted_chunk_ids = sorted(
        chunk_id_to_score.keys(), key=lambda x: (chunk_id_to_score[x], x[0], x[1]), reverse=True
    )

    # 정렬 순서에 따라 새 score 할당
    for index, chunk_id in enumerate(sorted_chunk_ids):
        chunk_id_to_score[chunk_id] = 1 / (index + 1)

    # 최종 결과 준비
    final_results = []
    semantic_count = 0
    bm25_count = 0
    for chunk_id in sorted_chunk_ids[:k]:
        chunk_metadata = next(
            chunk
            for chunk in db.metadata
            if chunk["doc_id"] == chunk_id[0] and chunk["original_index"] == chunk_id[1]
        )
        is_from_semantic = chunk_id in ranked_chunk_ids
        is_from_bm25 = chunk_id in ranked_bm25_chunk_ids
        final_results.append(
            {
                "chunk": chunk_metadata,
                "score": chunk_id_to_score[chunk_id],
                "from_semantic": is_from_semantic,
                "from_bm25": is_from_bm25,
            }
        )

        if is_from_semantic and not is_from_bm25:
            semantic_count += 1
        elif is_from_bm25 and not is_from_semantic:
            bm25_count += 1
        else:  # 둘 모두에 있음
            semantic_count += 0.5
            bm25_count += 0.5

    return final_results, semantic_count, bm25_count


def evaluate_db_advanced(
    db: ContextualVectorDB,
    original_jsonl_path: str,
    k_values: list[int] = None,
    db_name: str = "Hybrid Search",
):
    """
    여러 k 값에서 hybrid search(semantic + BM25)를 평가하고 결과 형식을 정리합니다.

    Args:
        db: ContextualVectorDB instance
        original_jsonl_path: 평가 dataset 경로
        k_values: 평가할 k 값 목록(기본값: [5, 10, 20])
        db_name: 평가 표시용 이름

    Returns:
        k 값을 결과에 mapping하는 dict 및 출처별 내역
    """
    if k_values is None:
        k_values = [5, 10, 20]
    original_data = load_jsonl(original_jsonl_path)
    es_bm25 = create_elasticsearch_bm25_index(db)
    results = {}

    print(f"{'=' * 70}")
    print(f"Evaluation Results: {db_name}")
    print(f"{'=' * 70}\n")

    try:
        # Warm-up query
        warm_up_queries = original_data[:10]
        for query_item in warm_up_queries:
            _ = retrieve_advanced(query_item["query"], db, es_bm25, k_values[0])

        for k in k_values:
            print(f"Evaluating Pass@{k}...")

            total_score = 0
            total_semantic_count = 0
            total_bm25_count = 0
            total_results = 0

            for query_item in tqdm(original_data, desc=f"Pass@{k}"):
                query = query_item["query"]
                golden_chunk_uuids = query_item["golden_chunk_uuids"]

                golden_contents = []
                for doc_uuid, chunk_index in golden_chunk_uuids:
                    golden_doc = next(
                        (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid),
                        None,
                    )
                    if golden_doc:
                        golden_chunk = next(
                            (
                                chunk
                                for chunk in golden_doc["chunks"]
                                if chunk["index"] == chunk_index
                            ),
                            None,
                        )
                        if golden_chunk:
                            golden_contents.append(golden_chunk["content"].strip())

                if not golden_contents:
                    continue

                retrieved_docs, semantic_count, bm25_count = retrieve_advanced(
                    query, db, es_bm25, k
                )

                chunks_found = 0
                for golden_content in golden_contents:
                    for doc in retrieved_docs[:k]:
                        retrieved_content = doc["chunk"]["original_content"].strip()
                        if retrieved_content == golden_content:
                            chunks_found += 1
                            break

                query_score = chunks_found / len(golden_contents)
                total_score += query_score

                total_semantic_count += semantic_count
                total_bm25_count += bm25_count
                total_results += len(retrieved_docs)

            total_queries = len(original_data)
            average_score = total_score / total_queries
            pass_at_n = average_score * 100

            semantic_percentage = (
                (total_semantic_count / total_results) * 100 if total_results > 0 else 0
            )
            bm25_percentage = (total_bm25_count / total_results) * 100 if total_results > 0 else 0

            results[k] = {
                "pass_at_n": pass_at_n,
                "average_score": average_score,
                "total_queries": total_queries,
                "semantic_percentage": semantic_percentage,
                "bm25_percentage": bm25_percentage,
            }

            print(f"Pass@{k}: {pass_at_n:.2f}%")
            print(f"Semantic: {semantic_percentage:.1f}% | BM25: {bm25_percentage:.1f}%\n")

        # 요약 표 출력
        print(f"{'=' * 70}")
        print(f"{'Metric':<12} {'Pass Rate':<12} {'Score':<12} {'Semantic':<12} {'BM25':<12}")
        print(f"{'-' * 70}")
        for k in k_values:
            r = results[k]
            print(
                f"{'Pass@' + str(k):<12} {r['pass_at_n']:>10.2f}% {r['average_score']:>10.4f} "
                f"{r['semantic_percentage']:>10.1f}% {r['bm25_percentage']:>10.1f}%"
            )
        print(f"{'=' * 70}\n")

        return results

    finally:
        # Elasticsearch index 삭제
        if es_bm25.es_client.indices.exists(index=es_bm25.index_name):
            es_bm25.es_client.indices.delete(index=es_bm25.index_name)
            print(f"Deleted Elasticsearch index: {es_bm25.index_name}")

In [39]:
results = evaluate_db_advanced(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual BM25 Hybrid Search",
)

Created index: contextual_bm25_index
Evaluation Results: Contextual BM25 Hybrid Search

Evaluating Pass@5...


Pass@5: 100%|██████████| 248/248 [00:05<00:00, 41.79it/s]


Pass@5: 88.86%
Semantic: 54.6% | BM25: 45.4%

Evaluating Pass@10...


Pass@10: 100%|██████████| 248/248 [00:05<00:00, 42.20it/s]


Pass@10: 92.31%
Semantic: 57.6% | BM25: 42.4%

Evaluating Pass@20...


Pass@20: 100%|██████████| 248/248 [00:05<00:00, 42.15it/s]


Pass@20: 95.23%
Semantic: 60.8% | BM25: 39.2%

Metric       Pass Rate    Score        Semantic     BM25        
----------------------------------------------------------------------
Pass@5            88.86%     0.8886       54.6%       45.4%
Pass@10           92.31%     0.9231       57.6%       42.4%
Pass@20           95.23%     0.9523       60.8%       39.2%

Deleted Elasticsearch index: contextual_bm25_index


## Reranking

Hybrid search로 뛰어난 결과(93.21% Pass@10)를 얻었지만 성능을 더 끌어낼 기법이 하나 더 있습니다. **Reranking**입니다.

### Reranking이란 무엇인가요?

Reranking은 2단계 retrieval 방식입니다:

1. **1단계 - 폭넓은 Retrieval**: 필요한 것보다 많은 후보를 retrieval하여 넓게 탐색(예: chunk 100개 retrieval)
2. **2단계 - 정확한 선택**: 전문 reranking 모델로 후보를 채점하여 가장 관련성 높은 상위 k개만 선택

**왜 작동하나요?** 초기 retrieval 방법(embedding, BM25)은 수백만 문서에서의 속도에 최적화되어 있습니다. Reranking 모델은 느리지만 더 정확하며 작은 후보 집합을 깊이 분석할 수 있습니다. 실제로 잘 작동하는 속도/정확도 trade-off를 만듭니다.

### Reranking 접근 방식

이 예제에서는 전체 hybrid search가 아닌 contextual embedding만을 기반으로 한 더 단순한 reranking pipeline을 사용합니다. 과정은 다음과 같습니다:

1. **추가 Retrieval**: 필요한 것보다 10배 많은 결과 조회(예: 10개가 필요할 때 chunk 100개 retrieval)
2. **Cohere로 Reranking**: Cohere의 `rerank-english-v3.0` 모델로 모든 후보 채점
3. **Top-k 선택**: 가장 높은 score의 결과만 반환

Reranking 모델은 원래 chunk 내용과 생성한 context 설명에 모두 접근하여 정확한 관련성 판단을 위한 풍부한 정보를 얻습니다.

### 예상 성능

Reranking은 작지만 의미 있는 개선을 제공합니다:
- **Reranking 없이**: 92.34% Pass@10(contextual embedding만)
- **Reranking 사용**: 약 95% Pass@10(추가 2-3% 향상)

작아 보일 수 있지만 production 시스템에서 실패를 7.66%에서 약 5%로 줄이면 사용자 경험을 크게 개선할 수 있습니다. Trade-off는 query latency입니다. Reranking은 후보 집합 크기에 따라 query당 약 100-200ms를 추가합니다.

In [ ]:
import json
from collections.abc import Callable
from typing import Any

import cohere
from tqdm import tqdm


def evaluate_db_rerank(
    db, original_jsonl_path: str, k_values: list[int] = None, db_name: str = "Reranking"
):
    """
    여러 k 값에서 reranking 성능을 평가하고 결과 형식을 정리합니다.

    Args:
        db: ContextualVectorDB instance
        original_jsonl_path: 평가 dataset 경로
        k_values: 평가할 k 값 목록(기본값: [5, 10, 20])
        db_name: 평가 표시용 이름

    Returns:
        k 값을 결과에 mapping하는 dict
    """
    if k_values is None:
        k_values = [5, 10, 20]
    original_data = load_jsonl(original_jsonl_path)
    co = cohere.Client(os.getenv("COHERE_API_KEY"))
    results = {}

    print(f"{'=' * 60}")
    print(f"Evaluation Results: {db_name}")
    print(f"{'=' * 60}\n")

    for k in k_values:
        print(f"Evaluating Pass@{k} with reranking...")

        total_score = 0
        total_queries = len(original_data)

        for query_item in tqdm(original_data, desc=f"Pass@{k}"):
            query = query_item["query"]
            golden_chunk_uuids = query_item["golden_chunk_uuids"]

            # Golden content 찾기
            golden_contents = []
            for doc_uuid, chunk_index in golden_chunk_uuids:
                golden_doc = next(
                    (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid), None
                )
                if golden_doc:
                    golden_chunk = next(
                        (chunk for chunk in golden_doc["chunks"] if chunk["index"] == chunk_index),
                        None,
                    )
                    if golden_chunk:
                        golden_contents.append(golden_chunk["content"].strip())

            if not golden_contents:
                continue

            # Retrieval 및 reranking
            semantic_results = db.search(query, k=k * 10)

            # Reranking용 문서 준비
            documents = [
                f"{res['metadata']['original_content']}\n\nContext: {res['metadata']['contextualized_content']}"
                for res in semantic_results
            ]

            # Reranking
            rerank_response = co.rerank(
                model="rerank-english-v3.0", query=query, documents=documents, top_n=k
            )
            time.sleep(0.1)  # Rate limiting

            # 최종 결과 조회
            retrieved_docs = []
            for r in rerank_response.results:
                original_result = semantic_results[r.index]
                retrieved_docs.append(
                    {"chunk": original_result["metadata"], "score": r.relevance_score}
                )

            # 결과에 golden chunk가 있는지 확인
            chunks_found = 0
            for golden_content in golden_contents:
                for doc in retrieved_docs[:k]:
                    retrieved_content = doc["chunk"]["original_content"].strip()
                    if retrieved_content == golden_content:
                        chunks_found += 1
                        break

            query_score = chunks_found / len(golden_contents)
            total_score += query_score

        average_score = total_score / total_queries
        pass_at_n = average_score * 100

        results[k] = {
            "pass_at_n": pass_at_n,
            "average_score": average_score,
            "total_queries": total_queries,
        }

        print(f"Pass@{k}: {pass_at_n:.2f}%")
        print(f"Average Score: {average_score:.4f}\n")

    # 요약 표 출력
    print(f"{'=' * 60}")
    print(f"{'Metric':<15} {'Pass Rate':<15} {'Score':<15}")
    print(f"{'-' * 60}")
    for k in k_values:
        pass_rate = f"{results[k]['pass_at_n']:.2f}%"
        score = f"{results[k]['average_score']:.4f}"
        print(f"{'Pass@' + str(k):<15} {pass_rate:<15} {score:<15}")
    print(f"{'=' * 60}\n")

    return results

In [48]:
results = evaluate_db_rerank(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual Embeddings + Reranking",
)

Evaluation Results: Contextual Embeddings + Reranking

Evaluating Pass@5 with reranking...


Pass@5: 100%|██████████| 248/248 [01:40<00:00,  2.47it/s]


Pass@5: 92.15%
Average Score: 0.9215

Evaluating Pass@10 with reranking...


Pass@10: 100%|██████████| 248/248 [02:29<00:00,  1.66it/s]


Pass@10: 95.26%
Average Score: 0.9526

Evaluating Pass@20 with reranking...


Pass@20: 100%|██████████| 248/248 [03:03<00:00,  1.35it/s]

Pass@20: 97.45%
Average Score: 0.9745

Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          92.15%          0.9215         
Pass@10         95.26%          0.9526         
Pass@20         97.45%          0.9745         



 
Reranking은 가장 강력한 결과를 제공하여 retrieval 실패를 거의 없앱니다. 각 기법이 이전 기법 위에 어떻게 쌓여 개선을 이루었는지 살펴보겠습니다.

Baseline RAG 시스템의 87% Pass@10에서 시작하여 고급 retrieval 기법을 체계적으로 적용하면서 95%를 넘었습니다. 각 방법은 서로 다른 약점을 해결합니다. Contextual embedding은 "고립된 chunk" 문제를 해결하고, hybrid search는 embedding이 놓치는 keyword별 query를 포착하며, reranking은 더 정교한 relevance scoring으로 최종 선택을 개선합니다.

| 접근 방식 | Pass@5 | Pass@10 | Pass@20 |
|----------|--------|---------|---------|
| **Baseline RAG** | 80.92% | 87.15% | 90.06% |
| **+ Contextual Embedding** | 88.12% | 92.34% | 94.29% |
| **+ Hybrid Search (BM25)** | 86.43% | 93.21% | 94.99% |
| **+ Reranking** | 92.15% | 95.26% | 97.45% |

**핵심 사항:**

1. **Contextual embedding이 가장 큰 단일 개선**(+5-7 percentage point)을 제공하여 chunk에 문서 수준 context를 추가하면 retrieval 품질이 크게 개선됨을 확인했습니다. 이 기법만으로 최적 성능까지의 90%를 달성합니다.

2. **Reranking은 절대적으로 가장 높은 성능**인 95.26% Pass@10을 달성합니다. Query의 95%에서 상위 10개 결과에 올바른 chunk가 나타난다는 의미입니다. Baseline RAG 대비 **retrieval 실패 47% 감소**를 나타냅니다(실패율 12.85%에서 4.74%로 감소).

3. **Trade-off가 중요합니다**: 각 기법은 복잡성과 비용을 추가합니다:
   - Contextual embedding: 일회성 ingestion 비용(이 dataset에서 prompt caching 사용 시 약 $3)
   - Hybrid search: Elasticsearch infrastructure 및 유지 관리 필요
   - Reranking: Query latency 100-200ms 및 query별 API 비용 추가(query당 약 $0.002)

4. 요구 사항에 따라 **접근 방식을 선택하세요**:
   - **대량 처리, 비용에 민감**: Contextual embedding만(92% Pass@10, query별 비용 없음)
   - **최대 정확도, latency 허용**: 전체 reranking pipeline(95% Pass@10, 최고의 precision)
   - **균형 잡힌 production 시스템**: Query별 비용 없이 뛰어난 성능의 hybrid search(93% Pass@10)

대부분의 production RAG 시스템에서 **contextual embedding은 최고의 비용 대비 성능**을 제공하며 일회성 ingestion 비용만으로 92% Pass@10을 달성합니다. Precision이 추가로 2-3 percentage point 필요하고 infrastructure 또는 query 비용을 감당할 수 있다면 hybrid search 및 reranking을 사용할 수 있습니다.

### 다음 단계 및 핵심 사항

1) Contextual Embedding으로 retrieval 성능을 개선하는 방법을 보여준 뒤 Contextual BM25와 reranking으로 추가 개선을 제공했습니다.

2) 이 예제는 codebase를 사용했지만 내부 회사 knowledge base, 재무 및 법률 콘텐츠, 교육 콘텐츠 등 다른 데이터 유형에도 적용됩니다. 

3) AWS 사용자라면 `contextual-rag-lambda-function`의 Lambda 함수로 시작할 수 있으며 GCP 사용자라면 자신의 Cloud Run instance를 실행하여 비슷한 패턴을 따를 수 있습니다!
